In [ ]:
import os
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from google.colab import drive

# ==========================================
# 1. DATA PREPARATION
# ==========================================
drive.mount('/content/drive')

ZIP_PATH = '<Your drive zip path of your dataset>'
LOCAL_EXTRACT_PATH = '/content/dataset'

if not os.path.exists(LOCAL_EXTRACT_PATH):
    print("Unzipping dataset for speed...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_PATH)
    print("Unzip complete!")

TRAIN_DIR = os.path.join(LOCAL_EXTRACT_PATH, 'train')
TEST_DIR = os.path.join(LOCAL_EXTRACT_PATH, 'test')

# Constants
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 100
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms (Normalization is handled by ToTensor() scaling to [0,1])
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder(TRAIN_DIR, transform=transform)
test_data = datasets.ImageFolder(TEST_DIR, transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# ==========================================
# 2. MODEL ARCHITECTURES
# ==========================================

# --- Custom CNN ---
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten()
        )
        # Flat size calculation: (224-2)/2 = 111; (111-2)/2 = 54
        self.classifier = nn.Sequential(
            nn.Linear(64 * 54 * 54, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- VGG16 Transfer Learning ---
def get_vgg_model(num_classes):
    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for param in vgg.features.parameters():
        param.requires_grad = False  # Freeze backbone

    # Replace the final layer
    in_features = vgg.classifier[6].in_features
    vgg.classifier[6] = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return vgg

# ==========================================
# 3. TRAINING UTILITY
# ==========================================

def train_engine(model, train_loader, val_loader, epochs=10):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = []

    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation phase
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        history.append(accuracy)
        print(f"Epoch {epoch+1}/{epochs} | Val Acc: {accuracy:.4f}")

    return history

# ==========================================
# 4. EXECUTION & VISUALIZATION
# ==========================================

print("\n--- Training Custom Model ---")
custom_net = CustomCNN(NUM_CLASSES)
h_custom = train_engine(custom_net, train_loader, test_loader, EPOCHS)

print("\n--- Training VGG16 Model ---")
vgg_net = get_vgg_model(NUM_CLASSES)
h_vgg = train_engine(vgg_net, train_loader, test_loader, EPOCHS)

# Plot Results
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), h_custom, label='Custom CNN', marker='o')
plt.plot(range(1, EPOCHS+1), h_vgg, label='VGG16 Transfer', marker='s')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Save Weights
torch.save(custom_net.state_dict(), 'custom_model.pth')
torch.save(vgg_net.state_dict(), 'vgg16_model.pth')
print("Training Complete. Weights saved as .pth files.")

# ==========================================
# PART 3: EVALUATION & CONFUSION MATRIX
# ==========================================

import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. Ensure Model is on GPU and Class Names exist
# We use 'vgg_net' as defined in the previous training block
model_to_eval = vgg_net
model_to_eval.to(DEVICE)
class_names = train_data.classes

print(f"Generating predictions for VGG16 on {len(class_names)} classes...")

y_true = []
y_pred = []

# 2. Get Predictions for the Test Set
model_to_eval.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient tracking for speed and memory
    for images, labels in test_loader:
        # Move data to the same device as the model (GPU)
        images = images.to(DEVICE)

        # Get model output
        outputs = model_to_eval(images)

        # Convert outputs to predicted class index
        _, predicted = torch.max(outputs, 1)

        # Move back to CPU and convert to numpy for Scikit-Learn
        y_true.extend(labels.numpy()) # Labels are already on CPU usually
        y_pred.extend(predicted.cpu().numpy())

# 3. Plot the Confusion Matrix
# Since 100 classes is huge, we set annot=False to avoid a mess of text
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
plt.title('VGG16 Confusion Matrix: Butterfly Species', fontsize=16)
plt.xlabel('Predicted Species Index', fontsize=12)
plt.ylabel('Actual Species Index', fontsize=12)
plt.show()

# 4. Print the Detailed Report
print("\n--- Detailed Classification Report ---")
# Using the class_names we got from the dataset earlier
# Note: If memory is an issue, we only print the first 10 classes or summary
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

# 5. Optional: Save the report to a text file for your project
with open("classification_report.txt", "w") as f:
    f.write(report)
print("\nReport saved as classification_report.txt")

# ==========================================
# PART 4: EVALUATING THE CUSTOM CNN (FIXED NAMES)
# ==========================================

import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. Setup names and storage
# We use 'train_data' and 'test_loader' defined in Part 1
class_names = train_data.classes
y_true_custom = []
y_pred_custom = []

# 2. Generate Predictions for Custom Model
print("Predicting with Custom CNN...")

# Using 'custom_net' (The variable name from our training block)
# If you get another NameError, run the training cell first!
custom_net.to(DEVICE)
custom_net.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        # Forward pass through custom_net
        outputs = custom_net(images)

        # Get predictions
        _, predicted = torch.max(outputs, 1)

        # Move to CPU for metrics
        y_true_custom.extend(labels.cpu().numpy())
        y_pred_custom.extend(predicted.cpu().numpy())

# 3. Generate the Confusion Matrix Graph
cm_custom = confusion_matrix(y_true_custom, y_pred_custom)

plt.figure(figsize=(24, 18))
sns.heatmap(cm_custom, annot=False, cmap='Reds', cbar=True)

plt.title('Custom CNN: Butterfly Species Confusion Matrix', fontsize=20)
plt.xlabel('Predicted Species Index', fontsize=14)
plt.ylabel('Actual Species Index', fontsize=14)
plt.show()

# 4. Print the Detailed Report
print("\n--- Detailed Classification Report: CUSTOM CNN ---")
print(classification_report(y_true_custom, y_pred_custom, target_names=class_names))

# ==========================================
# PART 5: FINE-TUNING & DATA AUGMENTATION
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# 1. STEP A: DATA AUGMENTATION (Applied to Training Only)
train_transform_augmented = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.1)),
    transforms.ToTensor(),
    # Standard ImageNet normalization for VGG16
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Use continuative TRAIN_DIR and BATCH_SIZE
train_data_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform_augmented)
train_loader_aug = DataLoader(train_data_aug, batch_size=32, shuffle=True, num_workers=2)

# 2. STEP B: BUILD THE FINE-TUNING MODEL
print("Initializing Fine-Tuned Model...")
tuned_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Freeze all layers first
for param in tuned_model.parameters():
    param.requires_grad = False

# Unfreeze the last block of VGG16 (Block 5 starts around layer 24)
for param in tuned_model.features[24:].parameters():
    param.requires_grad = True

# 3. New Classifier Head
n_inputs = tuned_model.classifier[0].in_features
tuned_model.classifier = nn.Sequential(
    nn.Linear(n_inputs, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 100) # 100 Butterfly classes
)

tuned_model = tuned_model.to(DEVICE)

# 4. Optimizer: Differential Learning Rates
# We train the un-frozen base layers very slowly (1e-5)
# and the new head at a standard pace (1e-4)
optimizer = optim.Adam([
    {'params': tuned_model.features[24:].parameters(), 'lr': 1e-5},
    {'params': tuned_model.classifier.parameters(), 'lr': 1e-4}
])

criterion = nn.CrossEntropyLoss()

# 5. Training Loop
h_tuned_val_acc = []

def train_tuned():
    print("\n--- Starting Fine-Tuning (Epochs 1-10) ---")
    for epoch in range(10):
        tuned_model.train()
        for images, labels in train_loader_aug:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = tuned_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation after each epoch
        tuned_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader: # Uses your existing test_loader
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = tuned_model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        acc = correct / total
        h_tuned_val_acc.append(acc)
        print(f"Epoch {epoch+1}/10 | Val Acc: {acc:.4f}")

train_tuned()

# ==========================================
# PART 6: FINAL COMPARISON GRAPH
# ==========================================

plt.figure(figsize=(14, 7))

# Plotting the three stages of our experiment
# Note: We use the history variables 'h_custom' and 'h_vgg' from Part 2
plt.plot(h_custom, label='1. Custom CNN (Scratch)', color='gray', linestyle='--')
plt.plot(h_vgg, label='2. Standard VGG16 (Frozen)', color='blue')
plt.plot(h_tuned_val_acc, label='3. Fine-Tuned VGG16 (Unfrozen)', color='green', linewidth=3)

plt.title('The Evolution of Accuracy: Scratch vs. Transfer vs. Fine-Tuning', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)

# Visual lines for the "Accuracy Jump"
plt.axhline(y=max(h_vgg), color='blue', linestyle=':', alpha=0.3)
plt.axhline(y=max(h_tuned_val_acc), color='green', linestyle=':', alpha=0.3)

plt.show()

# Save the final masterpiece
torch.save(tuned_model.state_dict(), 'vgg16_finetuned.pth')
print("Fine-tuned model saved as vgg16_finetuned.pth")

# ==========================================
# PART 7: GRAD-CAM VISUALIZATION (EXPLAINABLE AI)
# ==========================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2 # Used for resizing the heatmap

# 1. DEFINE GRAD-CAM EXTRACTOR
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Hook to capture gradients
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        # Hook to capture activations
        def forward_hook(module, input, output):
            self.activations = output

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate_heatmap(self, input_tensor, class_idx):
        self.model.eval()
        output = self.model(input_tensor)

        # Zero gradients
        self.model.zero_grad()

        # Target specific class
        loss = output[0, class_idx]
        loss.backward()

        # Pool the gradients across the channels
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])

        # Weight the activations by the pooled gradients
        for i in range(self.activations.shape[1]):
            self.activations[:, i, :, :] *= pooled_gradients[i]

        # Average the channels to get the heatmap
        heatmap = torch.mean(self.activations, dim=1).squeeze()

        # ReLU on heatmap (we only care about positive influence)
        heatmap = F.relu(heatmap)

        # Normalize to [0, 1]
        heatmap /= torch.max(heatmap)

        return heatmap.cpu().detach().numpy()

# 2. INITIALIZE EXTRACTOR
# For VGG16, the last conv layer is features[28]
cam_extractor = GradCAM(tuned_model, tuned_model.features[28])

# 3. VISUALIZATION LOOP
# Grab a larger batch to ensure variety
images, labels = next(iter(DataLoader(test_data, batch_size=50, shuffle=True)))

unique_indices = []
seen_labels = set()

for i in range(len(labels)):
    label_id = labels[i].item()
    if label_id not in seen_labels:
        unique_indices.append(i)
        seen_labels.add(label_id)
    if len(unique_indices) == 5:
        break

num_to_show = len(unique_indices)
plt.figure(figsize=(12, num_to_show * 4))

for idx, img_idx in enumerate(unique_indices):
    img_tensor = images[img_idx:img_idx+1].to(DEVICE)
    label = labels[img_idx].item()

    # Generate Heatmap
    heatmap = cam_extractor.generate_heatmap(img_tensor, label)

    # Resize heatmap to 224x224 to match original image
    heatmap = cv2.resize(heatmap, (224, 224))

    # Convert image back to displayable format (Undo ImageNet Normalization)
    img_show = images[img_idx].permute(1, 2, 0).cpu().numpy()
    img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img_show = np.clip(img_show, 0, 1)

    # Plot Original
    plt.subplot(num_to_show, 2, 2*idx + 1)
    plt.imshow(img_show)
    plt.title(f"Species: {class_names[label]}", fontsize=10)
    plt.axis('off')

    # Plot Grad-CAM
    plt.subplot(num_to_show, 2, 2*idx + 2)
    plt.imshow(img_show)
    plt.imshow(heatmap, cmap='jet', alpha=0.4) # Overlay heatmap
    plt.title("AI Focus (Grad-CAM)", fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

# ==========================================
# PART 8: FINAL EVALUATION (FINE-TUNED MODEL)
# ==========================================

import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. Gather Predictions
print("Generating predictions for the Fine-Tuned VGG16...")
y_true = []
y_pred = []

# Ensure tuned_model is on the device and in eval mode
tuned_model.to(DEVICE)
tuned_model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        # Move images to GPU
        images = images.to(DEVICE)

        # Forward pass
        outputs = tuned_model(images)

        # Get the index of the highest logit
        _, predicted = torch.max(outputs, 1)

        # Move back to CPU for Scikit-Learn metrics
        # Labels are typically already on CPU from the DataLoader,
        # but .cpu().numpy() is safer.
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# 2. Calculate Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# 3. Plotting
plt.figure(figsize=(24, 20)) # Large size for 100 species
sns.heatmap(
    cm,
    annot=False,
    cmap='Greens',
    cbar=True,
    xticklabels=False, # Hidden to prevent 100 labels from overlapping
    yticklabels=False
)

plt.title('Confusion Matrix: Fine-Tuned VGG16 (Butterfly Species)', fontsize=20)
plt.xlabel('Predicted Species Index', fontsize=14)
plt.ylabel('Actual Species Index', fontsize=14)
plt.show()

# 4. Final Verification Report
print("\n--- Final Classification Report ---")
# Using the continuative class_names from train_data.classes
print(classification_report(y_true, y_pred, target_names=class_names))

# Save the final report to a file
final_report = classification_report(y_true, y_pred, target_names=class_names)
with open("fine_tuned_vgg16_report.txt", "w") as f:
    f.write(final_report)

print("\nSuccess! The final report is saved as 'fine_tuned_vgg16_report.txt'.")

# ==========================================
# PART 9: THE IMPROVED CUSTOM CNN (OPTIMIZED)
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import matplotlib.pyplot as plt

# --- 1. BUILD THE IMPROVED ARCHITECTURE ---
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=100):
        super(ImprovedCNN, self).__init__()

        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f), # Standardizes inputs to each layer
                nn.ReLU(),
                nn.MaxPool2d(2) # Downsamples spatial dimensions
            )

        self.block1 = conv_block(3, 32)   # Output: 32 x 112 x 112
        self.block2 = conv_block(32, 64)  # Output: 64 x 56 x 56
        self.block3 = conv_block(64, 128) # Output: 128 x 28 x 28
        self.block4 = nn.Sequential(      # Output: 256 x 28 x 28
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        # GAP reduces any spatial size to 1x1, making the model more robust to image size
        self.gap = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        return self.classifier(x)

# Initialize using our continuative variables
improved_model = ImprovedCNN(NUM_CLASSES).to(DEVICE)

# --- 2. OPTIMIZER & SCHEDULER ---
optimizer = optim.Adam(improved_model.parameters(), lr=0.001)
# Reduces learning rate when accuracy plateaus
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()

# --- 3. TRAINING LOOP ---
h_improved_acc = []

print("\n" + "="*50)
print("STARTING TRAINING: IMPROVED CUSTOM CNN")
print("="*50)

for epoch in range(20):
    improved_model.train()
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = improved_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation
    improved_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = improved_model(images)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()

    val_acc = correct / total
    h_improved_acc.append(val_acc)

    # Update scheduler based on accuracy
    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/20 - Val Acc: {val_acc:.4f} | LR: {current_lr}")

# --- 4. FINAL COMPARATIVE VISUALIZATION ---
plt.figure(figsize=(14, 7))

# Plotting the progression using previous session variables
if 'h_custom' in locals():
    plt.plot(h_custom, label='1. Original Custom (Baseline)', color='red', linestyle='--')

plt.plot(h_improved_acc, label='2. Improved Custom (Batchnorm + GAP)', color='orange', linewidth=2, marker='x')

if 'h_tuned_val_acc' in locals():
    plt.plot(h_tuned_val_acc, label='3. Fine-Tuned VGG16 (Champion)', color='green', linewidth=3, marker='o')

plt.title('Performance Race: Custom vs. Optimized vs. Transfer Learning', fontsize=16)
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

torch.save(improved_model.state_dict(), 'improved_custom_model.pth')

# ==========================================
# PART 10: GRAD-CAM FOR IMPROVED CUSTOM CNN
# ==========================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torch.utils.data import DataLoader

# 1. DEFINE GRAD-CAM FOR CUSTOM ARCHITECTURE
class GradCAMCustom:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Using register_full_backward_hook for better compatibility in PyTorch 2.x+
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_image, class_idx):
        self.model.zero_grad()
        output = self.model(input_image)

        # Target the specific class score
        loss = output[0, class_idx]
        loss.backward()

        # Weight activations by the mean of the gradients (GAP over gradients)
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)

        # Apply ReLU to keep only features that have a positive influence on the class
        cam = F.relu(cam)

        # Move to CPU and normalize
        cam = cam.cpu().detach().numpy().squeeze()
        cam = cv2.resize(cam, (224, 224))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# 2. INITIALIZE EXTRACTOR
# We target block4[0] which is the Conv2d layer in your improved architecture
target_layer_custom = improved_model.block4[0]
cam_extractor_custom = GradCAMCustom(improved_model, target_layer_custom)

# 3. SELECTION & VISUALIZATION
# Get a batch from the test_data (continuative)
images, labels = next(iter(DataLoader(test_data, batch_size=60, shuffle=True)))

unique_indices = []
seen_labels = set()
for i in range(len(labels)):
    if labels[i].item() not in seen_labels:
        unique_indices.append(i)
        seen_labels.add(labels[i].item())
    if len(unique_indices) == 5: break

plt.figure(figsize=(12, 20))

for idx, img_idx in enumerate(unique_indices):
    img_tensor = images[img_idx:img_idx+1].to(DEVICE)
    label = labels[img_idx].item()

    # Generate heatmap for the Improved Custom Model
    heatmap = cam_extractor_custom.generate_heatmap(img_tensor, label)

    # Undo Normalization for display
    img_show = images[img_idx].permute(1, 2, 0).cpu().numpy()
    img_show = (img_show * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img_show = np.clip(img_show, 0, 1)

    # Original Image
    plt.subplot(5, 2, 2*idx + 1)
    plt.imshow(img_show)
    plt.title(f"Actual: {class_names[label]}")
    plt.axis('off')

    # Heatmap Overlay
    plt.subplot(5, 2, 2*idx + 2)
    plt.imshow(img_show)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.title("Custom CNN: Activation Map")
    plt.axis('off')

plt.tight_layout()
plt.show()

def force_purge_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        if hasattr(module, '_backward_pre_hooks'):
            module._backward_pre_hooks.clear()
    print(f"Purge complete for {model.__class__.__name__}")

force_purge_hooks(improved_model)
force_purge_hooks(tuned_model)

import os
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

# 1. SETUP & DEVICE
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 100

# Ensure class_names is available
if 'train_data' in locals():
    class_names = train_data.classes
else:
    class_names = [f"Species {i}" for i in range(100)]

# 2. DEFINE ARCHITECTURES
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=100):
        super(ImprovedCNN, self).__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.block1 = conv_block(3, 32)
        self.block2 = conv_block(32, 64)
        self.block3 = conv_block(64, 128)
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        return self.classifier(x)

# 3. INITIALIZE MODELS & LOAD WEIGHTS
improved_model = ImprovedCNN(NUM_CLASSES).to(DEVICE)

tuned_model = models.vgg16(weights=None)
n_inputs = tuned_model.classifier[0].in_features
tuned_model.classifier = nn.Sequential(
    nn.Linear(n_inputs, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 100)
)
tuned_model = tuned_model.to(DEVICE)

# --- THE FIX: DISABLE INPLACE RELU FOR GRAD-CAM ---
def disable_inplace_relu(model):
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False

disable_inplace_relu(improved_model)
disable_inplace_relu(tuned_model)

# Load saved weights
try:
    if os.path.exists('improved_custom_model.pth'):
        improved_model.load_state_dict(torch.load('improved_custom_model.pth', map_location=DEVICE))
        print("✅ Custom Model loaded.")
    if os.path.exists('vgg16_finetuned.pth'):
        tuned_model.load_state_dict(torch.load('vgg16_finetuned.pth', map_location=DEVICE))
        print("✅ VGG16 Tuned loaded.")
except Exception as e:
    print(f"❌ Load Error: {e}")

# 4. REFINED GRAD-CAM
def get_gradcam_final(model, target_layer, img_tensor, class_idx):
    data = {"act": None, "grad": None}
    def f_hook(module, input, output): data["act"] = output.detach()
    def b_hook(module, grad_input, grad_output): data["grad"] = grad_output[0].detach()

    h1 = target_layer.register_forward_hook(f_hook)
    h2 = target_layer.register_full_backward_hook(b_hook)

    try:
        model.zero_grad()
        output = model(img_tensor)
        loss = output[0, class_idx]
        loss.backward()

        weights = torch.mean(data["grad"], dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * data["act"], dim=1, keepdim=True)
        cam = torch.relu(cam).cpu().numpy().squeeze()
        cam = cv2.resize(cam, (224, 224))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    finally:
        h1.remove()
        h2.remove()

# 5. EXECUTION
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def compare_visuals(img_path):
    img_pil = Image.open(img_path).convert('RGB')
    input_tensor = preprocess(img_pil).unsqueeze(0).to(DEVICE)

    improved_model.eval()
    tuned_model.eval()

    # Predictions
    with torch.no_grad():
        out_c = improved_model(input_tensor)
        out_v = tuned_model(input_tensor)
        res_c = torch.max(torch.softmax(out_c[0], 0), 0)
        res_v = torch.max(torch.softmax(out_v[0], 0), 0)

    # Heatmaps
    hm_c = get_gradcam_final(improved_model, improved_model.block4[0], input_tensor, res_c[1])
    hm_v = get_gradcam_final(tuned_model, tuned_model.features[28], input_tensor, res_v[1])

    # Plot
    plt.figure(figsize=(18, 6))
    raw_img = np.array(img_pil.resize((224, 224)))

    display_data = [
        (raw_img, "Original Image", None),
        (hm_c, f"Custom CNN: {class_names[res_c[1]]} ({res_c[0].item()*100:.1f}%)", 'jet'),
        (hm_v, f"VGG16 Tuned: {class_names[res_v[1]]} ({res_v[0].item()*100:.1f}%)", 'jet')
    ]

    for i, (img, title, cmap) in enumerate(display_data):
        plt.subplot(1, 3, i+1)
        plt.imshow(raw_img)
        if cmap:
            plt.imshow(img, cmap=cmap, alpha=0.4)
        plt.title(title)
        plt.axis('off')
    plt.show()

# 6. RUN
SAMPLE_IMG = "/content/mrgajowy3-butterfly-9089187_1920.jpg"
compare_visuals(SAMPLE_IMG)

import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
import os

def compare_models_on_image(image_path, model_custom, model_vgg, class_names):
    # 1. Verification & Loading
    if not os.path.exists(image_path):
        print(f"❌ Error: Image not found at {image_path}")
        return

    img = Image.open(image_path).convert('RGB')

    # Standardize preprocessing to match training
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Prepare tensor (unsqueeze adds the 'batch' dimension)
    input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)

    # 2. Inference: Improved Custom CNN
    model_custom.eval()
    with torch.no_grad():
        out_custom = model_custom(input_tensor)
        prob_custom = torch.nn.functional.softmax(out_custom[0], dim=0)
        conf_custom, pred_custom = torch.max(prob_custom, 0)

    # 3. Inference: Fine-Tuned VGG16
    model_vgg.eval()
    with torch.no_grad():
        out_vgg = model_vgg(input_tensor)
        prob_vgg = torch.nn.functional.softmax(out_vgg[0], dim=0)
        conf_vgg, pred_vgg = torch.max(prob_vgg, 0)

    # 4. Visualization
    plt.figure(figsize=(10, 8))

    # Show Original Image (resized for the plot)
    plt.imshow(img)
    plt.title("Butterfly Species Identification Test", fontsize=16, pad=20)
    plt.axis('off')

    # Create the result summary box
    result_text = (
        f"TEST RESULTS\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"CUSTOM CNN Prediction:  {class_names[pred_custom]}\n"
        f"Confidence Score:       {conf_custom*100:.2f}%\n"
        f"--------------------------\n"
        f"VGG16 TUNED Prediction: {class_names[pred_vgg]}\n"
        f"Confidence Score:       {conf_vgg*100:.2f}%"
    )

    # Place text below the image
    plt.figtext(0.5, 0.05, result_text, wrap=True, horizontalalignment='center',
                fontsize=12, family='monospace',
                bbox={'facecolor':'white', 'edgecolor':'orange', 'alpha':0.8, 'pad':15})

    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.show()

# --- EXECUTE ---
# Note: Update this path to the image you uploaded in the sidebar
test_image = "/content/mrgajowy3-butterfly-9089187_1920.jpg"

compare_models_on_image(
    test_image,
    improved_model,
    tuned_model,
    class_names
)

# ==========================================
# PART 12: DEEP ERROR ANALYSIS (TOP LOSS)
# ==========================================

import torch
import numpy as np
import matplotlib.pyplot as plt

def run_error_analysis(model, loader, class_names, num_images=8):
    """
    Identifies and visualizes images that the model found most difficult
    by calculating the Cross Entropy Loss for every single image.
    """
    model.eval()
    all_losses = []
    all_preds = []
    all_labels = []
    all_images = []

    # Use reduction='none' to keep the loss values individual per image
    criterion = torch.nn.CrossEntropyLoss(reduction='none')

    print(f"Analyzing errors for {model.__class__.__name__}...")

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)

            # Calculate loss per sample
            loss = criterion(outputs, labels)

            # Convert to probabilities for confidence tracking
            probs = torch.nn.functional.softmax(outputs, dim=1)
            conf, preds = torch.max(probs, 1)

            all_losses.extend(loss.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(images.cpu())

    # Sort indices by highest loss (descending)
    sorted_idx = np.argsort(all_losses)[::-1]

    # Visualization
    plt.figure(figsize=(16, 10))
    for i in range(min(num_images, len(sorted_idx))):
        idx = sorted_idx[i]
        plt.subplot(2, 4, i+1)

        # Un-normalize image for plotting
        img = all_images[idx].permute(1, 2, 0).numpy()
        img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)

        plt.imshow(img)

        # Color coding: Red for wrong, Green for correct (but high loss)
        is_correct = (all_labels[idx] == all_preds[idx])
        text_color = 'green' if is_correct else 'red'

        title = (f"True: {class_names[all_labels[idx]]}\n"
                 f"Pred: {class_names[all_preds[idx]]}\n"
                 f"Loss: {all_losses[idx]:.2f}")

        plt.title(title, color=text_color, fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)
    plt.suptitle(f"Error Analysis: Top Struggles for {model.__class__.__name__}", fontsize=16)
    plt.show()

# --- RUN ANALYSIS ---
# Running for the Custom CNN
run_error_analysis(improved_model, test_loader, class_names)

# Running for the VGG16 (Tuned)
run_error_analysis(tuned_model, test_loader, class_names)

# ==========================================
# PART 13: PRECISION-RECALL COMPARISON
# ==========================================

from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F

def plot_precision_recall_comparison(models, loader, names):
    plt.figure(figsize=(10, 8))
    colors = ['#FF8C00', '#228B22'] # Orange for Custom, Forest Green for VGG

    print("Generating Precision-Recall data (this may take a moment)...")

    for i, model in enumerate(models):
        model.eval()
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for images, labels in loader:
                images = images.to(DEVICE)
                outputs = model(images)
                # We need the probabilities for the PR curve calculation
                probs = F.softmax(outputs, dim=1)

                all_probs.append(probs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        # Flatten the batch results
        y_true_flat = np.concatenate(all_labels)
        y_score_flat = np.concatenate(all_probs)

        # Binarize labels for multi-class PR curve
        # We use micro-averaging to see global performance across 100 classes
        Y_test_bin = label_binarize(y_true_flat, classes=range(len(class_names)))

        # Calculate Micro-Average Precision-Recall
        # ravel() turns the matrices into single long vectors for global comparison
        precision, recall, _ = precision_recall_curve(Y_test_bin.ravel(), y_score_flat.ravel())
        avg_precision = average_precision_score(Y_test_bin, y_score_flat, average="micro")

        plt.plot(recall, precision, color=colors[i], lw=3,
                 label=f'{names[i]} (mAP = {avg_precision:.3f})')

    # Formatting the Plot
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("Recall (Ability to find all butterflies)")
    plt.ylabel("Precision (Ability to be correct when guessing)")
    plt.title("Precision-Recall Trade-off: Custom vs. VGG16", fontsize=14)
    plt.legend(loc="lower left", frameon=True, shadow=True)
    plt.grid(True, linestyle='--', alpha=0.6)

    # Add a baseline (chance level)
    baseline = 1 / len(class_names)
    plt.axhline(y=baseline, color='grey', linestyle=':', label='Random Guessing')

    plt.show()

# Run the final comparison
plot_precision_recall_comparison(
    [improved_model, tuned_model],
    test_loader,
    ["Improved Custom CNN", "Fine-Tuned VGG16"]
)

import gradio as gr
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# 1. Unified Prediction Function
def predict_butterfly_battle(img):
    """
    Takes an image, runs it through BOTH models,
    and returns top-5 results for each.
    """
    if img is None:
        return None, None

    # Ensure image is in PIL format for the preprocess pipeline
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)

    # Preprocess the image
    # Note: Ensure 'preprocess' and 'DEVICE' are defined in your environment
    input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)

    # 2. Inference Logic
    improved_model.eval()
    tuned_model.eval()

    with torch.no_grad():
        # Custom CNN Results
        out_custom = improved_model(input_tensor)
        probs_custom = F.softmax(out_custom[0], dim=0)
        top_probs_c, top_idxs_c = torch.topk(probs_custom, 5)
        res_custom = {class_names[top_idxs_c[i]]: float(top_probs_c[i]) for i in range(5)}

        # VGG16 Results
        out_vgg = tuned_model(input_tensor)
        probs_vgg = F.softmax(out_vgg[0], dim=0)
        top_probs_v, top_idxs_v = torch.topk(probs_vgg, 5)
        res_vgg = {class_names[top_idxs_v[i]]: float(top_probs_v[i]) for i in range(5)}

    return res_custom, res_vgg

# 3. Create the Unified Gradio Interface
with gr.Blocks(theme="glass", title="Butterfly Species Battle") as demo:
    gr.Markdown("# 🦋 Butterfly Species Battle: Custom vs. VGG16")
    gr.Markdown("Upload an image to see how a scratch-built CNN compares to a fine-tuned VGG16.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Input Image")
            btn = gr.Button("Compare Models", variant="primary")

        with gr.Column():
            output_vgg = gr.Label(num_top_classes=5, label="VGG16 (Transfer Learning)")
            output_custom = gr.Label(num_top_classes=5, label="Custom Improved CNN")

    # Connect the button to the function
    btn.click(
        fn=predict_butterfly_battle,
        inputs=input_img,
        outputs=[output_custom, output_vgg]
    )

    # Optional: Add examples for users to click
    gr.Examples(
        examples=["/content/mrgajowy3-butterfly-9089187_1920.jpg"],
        inputs=input_img
    )

# 4. Launch the application
demo.launch(share=True)